In [ ]:
import math, gc, torch, torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class Args: pass
args = Args()
args.data   = "merged_with_embeddings.parquet"
args.past   = 20       
args.future = 5        
args.batch  = 256
args.epochs = 20
args.lr     = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


class ECPriceDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x_price = torch.tensor(row.X_price, dtype=torch.float32)  # (past,1) 
        x_txt   = torch.tensor(row.passage_emb, dtype=torch.float32)  # passage vector
        y       = torch.tensor(row.y, dtype=torch.float32)  # (future,) 
        return x_price, x_txt, y


class DualModalCrossAttn(nn.Module):
    def __init__(self, d_price_in=1, d_txt=768, d_model=128, n_heads=4, seq_len=20, out_days=5):
        super().__init__()
        # 价格分支
        self.pos   = nn.Embedding(seq_len, d_model)
        self.p_proj= nn.Linear(d_price_in, d_model) # 
        self.p_attn= nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        # 文本分支
        self.t_proj= nn.Sequential(
            nn.Linear(d_txt, d_model), nn.ReLU(),
            nn.Linear(d_model, d_model))
        # Cross-Attention
        self.cross = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        # Transformer 融合
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, 4*d_model, batch_first=True)
        self.fuser = nn.TransformerEncoder(enc_layer, num_layers=2)
        # 输出
        self.head  = nn.Linear(d_model, out_days)

    def forward(self, price_seq, txt_emb):
        B, T, _ = price_seq.shape
        pos = self.pos(torch.arange(T, device=price_seq.device))[None]
        h_p = self.p_proj(price_seq) + pos                 # (B,T,d)
        h_p, _ = self.p_attn(h_p, h_p, h_p)                # 自注意力
        z_p = h_p.mean(dim=1, keepdim=True)                # 池化 (B,1,d)

        z_t = self.t_proj(txt_emb).unsqueeze(1)            # (B,1,d)
        z, _ = self.cross(z_p, z_t, z_t)                   # Cross-Attn
        z = self.fuser(z)                                  # Transformer 融合
        return self.head(z.squeeze(1))                     # (B,future)

# ==============================
# 样本构造（预测对数收益率）
# ==============================
def build_samples(df_raw, past=20, future=5):
    df = df_raw.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    samples = []
    for call_id, g in df.groupby("id"):
        g = g.sort_values("Date").reset_index(drop=True)

        prices = g["Price"].values.astype(float)
        logrets = np.diff(np.log(prices))  # 对数收益率, 长度 N-1
        emb = g["passage_emb"].values[0]

        for i in range(past, len(logrets) - future + 1):
            past_slice   = logrets[i-past:i]
            future_slice = logrets[i:i+future]
            samples.append({
                "passage_emb": emb,
                "X_price": past_slice.reshape(-1, 1),  # 输入收益率
                "y": future_slice                      # 预测收益率
            })
    return pd.DataFrame(samples)

# ==============================
# 训练/验证步骤
# ==============================
def step(loader, model, loss_fn, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total, total_loss = 0, 0.0
    with torch.set_grad_enabled(train):
        for x_p, x_t, y in loader:
            x_p, x_t, y = x_p.to(device), x_t.to(device), y.to(device)
            if train and scaler:
                with torch.autocast("cuda"):
                    pred = model(x_p, x_t)
                    loss = loss_fn(pred, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            elif train:
                pred = model(x_p, x_t)
                loss = loss_fn(pred, y)
                loss.backward(); optimizer.step(); optimizer.zero_grad()
            else:
                pred = model(x_p, x_t)
                loss = loss_fn(pred, y)
            total += y.size(0)
            total_loss += loss.item() * y.size(0)
    return total_loss / total

In [ ]:
df_raw = pd.read_parquet(args.data)

# 自动探测文本向量维度
d_txt = len(df_raw.loc[0, "passage_emb"])
print("Detected embedding dim:", d_txt)

# 构造样本
df_samples = build_samples(df_raw, args.past, args.future)
print("样本量:", len(df_samples))

# 划分训练/验证集
tr_df, val_df = train_test_split(df_samples, test_size=0.2, random_state=42)
train_dl = DataLoader(ECPriceDataset(tr_df), batch_size=args.batch, shuffle=True, drop_last=True)
val_dl   = DataLoader(ECPriceDataset(val_df), batch_size=args.batch)

# 初始化模型
model = DualModalCrossAttn(
    d_price_in = 1,
    d_txt      = d_txt,
    d_model    = 128,
    n_heads    = 4,
    seq_len    = args.past,
    out_days   = args.future
).to(device)

opt    = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
loss_f = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler() if device=="cuda" else None

best = math.inf
for epoch in range(1, args.epochs+1):
    tr_loss = step(train_dl, model, loss_f, opt, scaler)
    val_loss = step(val_dl, model, loss_f)
    if val_loss < best:
        best = val_loss
        torch.save(model.state_dict(), "best_cross_attn.pt")
    print(f"[{epoch:02d}] train {tr_loss:.6f} | val {val_loss:.6f} | best {best:.6f}")

print("Best Validation MSE:", best)

In [ ]:
def returns_to_prices(P_last, rets_pred, logret=True):
    """将预测的收益率序列还原为价格序列"""
    prices = []
    p = P_last
    for r in rets_pred:
        if logret:
            p = p * np.exp(r)
        else:
            p = p * (1 + r)
        prices.append(p)
    return np.array(prices)

# 使用示例（假设你已经有最后一天价格 last_price）
# model.load_state_dict(torch.load("best_cross_attn.pt"))
# model.eval()
# x_price, x_txt, _ = val_dataset[0]   # 取验证集第一条样本
# pred_rets = model(x_price.unsqueeze(0).to(device), x_txt.unsqueeze(0).to(device))
# pred_rets = pred_rets.detach().cpu().numpy().flatten()
# pred_prices = returns_to_prices(last_price, pred_rets, logret=True)
# print("预测未来价格:", pred_prices)